In [1]:
# Data Ingestion & Cleaning
import pandas as pd
import numpy as np
import glob
import os
import re

## Data Ingestion & Cleaning
<b> Main Functions: </b>
- merge_product_files(data_folder) → Reads multiple CSVs, merges them, removes duplicates.
- clean_products(df) → Cleans raw product info, extracts unit price, standardizes base unit price, and computes discounts.

In [199]:
def merge_product_files(data_folder='data'):
    """
    Merges all CSV files in the specified folder into a single DataFrame
    
    Args:
        data_folder (str): Path to folder containing CSV files
        
    Returns:
        pd.DataFrame: Combined product data with source file tracking
    """
    # Find all CSV files in the folder
    all_files = glob.glob(os.path.join(data_folder, "*.csv"))
    
    # Read and concatenate files
    dfs = []
    for file in all_files:
        df = pd.read_csv(file)
        df['source_file'] = os.path.basename(file)  # Track origin
        dfs.append(df)
    
    # Combine with duplicate handling
    combined_df = pd.concat(dfs, ignore_index=True)
    
    # Remove exact duplicates (same data from multiple files)
    combined_df.drop_duplicates(
        subset=['product_code'],  # Assuming this is your unique ID
        keep='first',
        inplace=True
    )
    
    return combined_df

In [206]:
import pandas as pd
import numpy as np
import re

def clean_products(df):
    """
    Clean and standardise Coles product data for Smart Cart modelling.

    BUSINESS RULES IMPLEMENTED:
    ---------------------------------------------------------
    1. If unit_price is empty → treat as '1ea'
    2. If item_price invalid (NaN, text, <=0):
         - replace with best_price if valid
         - otherwise set both item_price and best_price to NaN
    3. Add binary flags:
         - on_special = 1 if special_text is not empty
         - on_promotion = 1 if promo_text is not empty
    4. Standardise extract_date → YYYY-MM-DD
    5. Standardise unit_of_measure:
         - valid units kept (g, kg, ml, l, ea)
         - partial units auto-corrected ('1k' → '1kg', '10m' → '10ml')
         - invalid units → '1ea'
    6. unit_price_value recalculated for invalid units using item_price
    7. base_unit_price standardised per 100g/ml for model comparability
    """

    df = df.copy()

    # ---------------------------------------------------------
    # STEP 1 — Remove duplicates
    # ---------------------------------------------------------
    df.drop_duplicates(subset=["product_code"], inplace=True)

    # ---------------------------------------------------------
    # STEP 2 — Fix item_price (RULE 2)
    # ---------------------------------------------------------
    df["item_price"] = pd.to_numeric(df["item_price"], errors="coerce")
    df["best_price"] = pd.to_numeric(df["best_price"], errors="coerce")

    # If item_price invalid → replace with best_price
    invalid_item = df["item_price"].isna() | (df["item_price"] <= 0)
    df.loc[invalid_item, "item_price"] = df.loc[invalid_item, "best_price"]

    # If item_price STILL invalid → both become NaN
    still_invalid = df["item_price"].isna() | (df["item_price"] <= 0)
    df.loc[still_invalid, ["item_price", "best_price"]] = np.nan

    # ---------------------------------------------------------
    # STEP 3 — Add on_special & on_promotion flags
    # ---------------------------------------------------------
    df["on_special"] = df["special_text"].fillna("").str.strip().ne("").astype(int)
    df["on_promotion"] = df["promo_text"].fillna("").str.strip().ne("").astype(int)

    # ---------------------------------------------------------
    # STEP 4 — Standardise dates
    # ---------------------------------------------------------
    if "extract_date" in df.columns:
        df["extract_date"] = pd.to_datetime(df["extract_date"], errors="coerce").dt.date

    # ---------------------------------------------------------
    # STEP 5 — Extract price + unit from unit_price column
    # ---------------------------------------------------------
    def extract_price_unit(text):
        if pd.isna(text):
            return (np.nan, np.nan)

        text = str(text).lower()
            # Normalize "/" to "per"   →   "$4/100g" → "$4 per 100g"
        text = text.replace("/", " per ")
        
        price_match = re.search(r"\$([\d\.]+)", text)
        unit_match = re.search(r"per\s*([a-zA-Z0-9]+)", text)

        price = float(price_match.group(1)) if price_match else np.nan
        unit = unit_match.group(1) if unit_match else np.nan
        return (price, unit)

    price_unit_list = df["unit_price"].apply(extract_price_unit).tolist()
    df[["unit_price_value", "unit_of_measure"]] = pd.DataFrame(price_unit_list, index=df.index)

    #    If unit_price_value is NaN → fallback to item_price
    # ---------------------------------------------------------
    df["unit_price_value"] = df["unit_price_value"].fillna(df["item_price"])

    # Optional: if still NaN, fall back to best_price
    df["unit_price_value"] = df["unit_price_value"].fillna(df["best_price"])

    #  If item_price is NaN → fallback to unit_price_value

    df["item_price"] = df["item_price"].fillna(df["unit_price_value"])    
    df["best_price"] = df["best_price"].fillna(df["unit_price_value"])

    df['discount_percentage'] = np.where( df['item_price'] > df['best_price'], (df['item_price'] - df['best_price']) / df['item_price'], 0 )

    # ---------------------------------------------------------
    # STEP 6 — Robust unit cleaning (RULE 1 + invalid units)
    # ---------------------------------------------------------
    def clean_unit(u):
        """
        Accepts raw unit text, returns valid forms of:
        '100g', '1kg', '100ml', '1l', '1ea'
        Converts invalid units → '1ea'
        Fixes partial units: '1k'→'1kg', '10m'→'10ml', '1kgm'→'1kg'
        """

        if pd.isna(u):
            return "1ea"

        u = str(u).lower().strip()

        # Partial kilograms: "1k" → "1kg"
        if re.fullmatch(r"\d+k", u):
            return u + "g"

        # Partial millilitres: "10m" → "10ml"
        if re.fullmatch(r"\d+m", u):
            return u + "l"

        # Fix "1kgm"
        if u.endswith("kgm"):
            return u.replace("kgm", "kg")

        # Valid patterns: number + unit suffix
        if re.fullmatch(r"\d+(g|kg|ml|l|ea)", u):
            return u

        # Anything else → invalid → treat as 1 each
        return "1ea"

    df["unit_of_measure"] = df["unit_of_measure"].apply(clean_unit)

    # ---------------------------------------------------------
    # STEP 7 — Fix unit_price_value when invalid units → item_price
    # ---------------------------------------------------------
    # df["unit_price_value"] = np.where(
    #     df["unit_of_measure"] == "1ea",
    #     df["item_price"],
    #     df["unit_price_value"]
    # )

    # ---------------------------------------------------------
    # STEP 8 — Standardised base_unit_price (per 100g/100ml)
    # ---------------------------------------------------------
    def base_price(row):
        unit = row["unit_of_measure"]
        price = row["unit_price_value"]

        if pd.isna(price):
            return np.nan

        # Grams
        if unit.endswith("g") and not unit.endswith("kg"):
            num = float(unit.replace("g", ""))
            return price / (num / 100)

        # Kilograms
        if unit.endswith("kg"):
            num = float(unit.replace("kg", ""))
            return price / (num * 10)  # because per 100g

        # Millilitres
        if unit.endswith("ml"):
            num = float(unit.replace("ml", ""))
            return price / (num / 100)

        # Litres
        if unit.endswith("l"):
            num = float(unit.replace("l", ""))
            return price / (num * 10)  # per 100ml

        # Each
        if unit.endswith("ea"):
            return price

        return price  # fallback

    df["base_unit_price"] = df.apply(base_price, axis=1)

    # ---------------------------------------------------------
    # FINAL RETURN
    # ---------------------------------------------------------
    return df


In [207]:
folder_path = 'data/scrapped_data'
products_df = merge_product_files(folder_path)   

C:\Users\rayed\AppData\Local\Temp\ipykernel_22176\767959533.py:17: DtypeWarning: Columns (2,4,5,8,11,12,13,14,16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


In [208]:
products_df = clean_products(products_df)

In [209]:
products_df.columns

Index(['_id', 'product_code', 'category', 'item_name', 'best_price',
       'item_price', 'unit_price', 'special_text', 'promo_text', 'link',
       'extract_date', 'best_unit_price', 'price_was', 'timestamp',
       'category_slug', 'store', 'image', 'categories[0]', 'source_file',
       'on_special', 'on_promotion', 'unit_price_value', 'unit_of_measure',
       'discount_percentage', 'base_unit_price'],
      dtype='object')

In [210]:
products_df.isna().sum()

_id                        0
product_code               1
category                7037
item_name                 46
best_price              3418
item_price              3418
unit_price              4838
special_text           40493
promo_text             51513
link                       0
extract_date            4645
best_unit_price        36080
price_was              51199
timestamp              55329
category_slug          56275
store                      0
image                  50186
categories[0]          57148
source_file                0
on_special                 0
on_promotion               0
unit_price_value        3418
unit_of_measure            0
discount_percentage        0
base_unit_price         3418
dtype: int64

In [211]:
cols = ['product_code', 'category', 'item_name', 'extract_date','unit_price_value', 'unit_of_measure','base_unit_price', 'discount_percentage', 'on_special', 'on_promotion']
products_df_cleaned = products_df [cols] 


In [219]:
#products_df[products_df['unit_price'] == '$0.73 per 1ea']
products_df_cleaned[np.isnan(products_df_cleaned['unit_price_value'])]

,product_code,category,item_name,extract_date,unit_price_value,unit_of_measure,base_unit_price,discount_percentage,on_special,on_promotion


In [213]:
mask_blank_name = products_df_cleaned['item_name'].isna() | (products_df_cleaned['item_name'].str.strip() == "") 
#| products_df_remove_na['category'].isna() | (products_df_remove_na['category'].str.strip() == "")
products_df_cleaned = products_df_cleaned[~mask_blank_name]

In [214]:
# check null values
products_df_cleaned.isna().sum()

product_code              0
category               6991
item_name                 0
extract_date           4645
unit_price_value       3417
unit_of_measure           0
base_unit_price        3417
discount_percentage       0
on_special                0
on_promotion              0
dtype: int64

In [217]:
products_df_cleaned = products_df_cleaned.dropna(subset=["unit_price_value", "extract_date"])
products_df_cleaned.isna().sum()

product_code              0
category               6989
item_name                 0
extract_date              0
unit_price_value          0
unit_of_measure           0
base_unit_price           0
discount_percentage       0
on_special                0
on_promotion              0
dtype: int64

In [223]:
products_df_cleaned.to_csv('data/products.csv')